# mcc_trx_vs_merchant — сравнение MCC

Сравниваем два источника MCC на зерне **`n_agr × report_month`** (Jan–Jul 2026):

| Источник | Таблица | Смысл |
|----------|--------|--------|
| **Merchant** | `ods_alpha.scd1_merchants.n_mcc` | справочник точки/компании (как в `final_script_2`) |
| **Trx** | `ods_alpha.scd1_trx.n_mcc` | код с операции (+ терминал `c_nter`) |

## Зачем
- У мерчантов часто multiple MCC (~1600 кейсов в `final_df`).
- Нужно понять: **trx-слой даёт меньше multiple?** и насколько множества совпадают.
- `final_df` / DRP **пересобирать не нужно**.

## Гипотеза (проверяется цифрами в summary)
- Для дашборд-таблицы MCC (обороты / % эквайринга) → **trx**.
- Merchant MCC → справочно / drill-down.

## Outputs
- `mcc_compare_agr_month_2026_01_2026_07.csv`
- `mcc_compare_summary.csv`
- `mcc_compare_by_month_2026_01_2026_07.csv`


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

period_start = '2026-01-01'
period_end_exclusive = '2026-08-01'
period_months = pd.date_range(
    period_start,
    pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1),
    freq='MS',
)

output_dir = Path('/home/jovyan/documents/Equaring/Data')
if not output_dir.exists():
    output_dir = Path.cwd()
output_dir.mkdir(parents=True, exist_ok=True)

out_detail_csv = output_dir / 'mcc_compare_agr_month_2026_01_2026_07.csv'
out_summary_csv = output_dir / 'mcc_compare_summary.csv'

print('period_months =', [m.strftime('%Y-%m') for m in period_months])
print('output_dir =', output_dir)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')


## 1. Merchant MCC (`scd1_merchants`)

SA-договоры → все merchants компании (`n_cmp = n_cmp_client`) → distinct MCC на `n_agr`.

Merchant MCC **не зависит от месяца** (справочник), но для сравнения размножаем на каждый `report_month` периода (зерно agr×month).


In [ ]:
sql_merchant = """
with sa as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
)
select
  s.n_agr,
  s.agr_id,
  cast(m.n_mcc as string) as mcc
from sa s
left join ods_alpha.scd1_merchants m
  on cast(m.n_cmp as string) = s.n_cmp_client
where m.n_mcc is not null
  and trim(cast(m.n_mcc as string)) <> ''
"""

print('Loading merchant MCC map...')
with imp:
    merchant_raw = imp.fetch(sql_merchant)

if merchant_raw is None or len(merchant_raw) == 0:
    merchant_raw = pd.DataFrame(columns=['n_agr', 'agr_id', 'mcc'])

merchant_raw['n_agr'] = merchant_raw['n_agr'].astype(str)
merchant_raw['mcc'] = merchant_raw['mcc'].astype(str).str.strip()


def _join_unique(series):
    vals = sorted({
        str(v).strip()
        for v in series
        if v is not None and str(v).strip() not in ('', 'None', 'nan', 'NaN')
    })
    return ','.join(vals) if vals else None


merchant_by_agr = (
    merchant_raw.groupby('n_agr', as_index=False)
    .agg(
        agr_id=('agr_id', 'first'),
        mcc_merchant_list=('mcc', _join_unique),
        mcc_merchant_cnt=('mcc', lambda s: len({
            str(v).strip()
            for v in s
            if v is not None and str(v).strip() not in ('', 'None', 'nan', 'NaN')
        })),
    )
)

month_labels = [m.strftime('%Y-%m') for m in period_months]
merchant_month = (
    merchant_by_agr.assign(_k=1)
    .merge(pd.DataFrame({'report_month': month_labels, '_k': 1}), on='_k')
    .drop(columns='_k')
)

print('merchant agr with any MCC:', len(merchant_by_agr))
print('merchant agr with multi MCC:', int((merchant_by_agr['mcc_merchant_cnt'] > 1).sum()))
print('merchant_month rows:', len(merchant_month))
display(merchant_by_agr.sort_values('mcc_merchant_cnt', ascending=False).head(10))


## 2. Trx MCC (`scd1_trx.n_mcc`) — помесячно

Периметр как в `final_script_2` section 05 / `vd_acq_mcc_month.sql`:
SA + S01, RSHB acquirer, `c_nter` not null, не reject.

**Важно:** один запрос на весь Jan–Jul часто висит / жрёт 70–100+ GB.
Здесь цикл **по месяцу**: в SQL сразу агрегат `n_agr × mcc` (без выгрузки каждой trx-строки).

Если старый тяжёлый запрос ещё RUNNING в Impala — **Cancel** его в UI, затем запускайте эту ячейку.


In [ ]:
def _sql_trx_month(month_start: str, month_end_exclusive: str) -> str:
    """One calendar month; SQL returns agr×mcc aggregates (light)."""
    return f"""
with fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
sa_agr as (
  select distinct cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_mcc as string) as mcc,
    cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('{month_end_exclusive}' as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and t.n_mcc is not null
),
trx_base as (
  select n_trx, max(mcc) as mcc, max(n_amt_src) as n_amt_src
  from trx_base_raw
  group by n_trx
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
  group by cast(a.n_trx as string), cast(a.n_agr as string)
)
select
  ta.n_agr,
  tb.mcc,
  count(distinct tb.n_trx) as trx_cnt,
  sum(tb.n_amt_src) as trx_sum,
  sum(ta.n_amt_tax) as commission_from_ops
from trx_base tb
join ta on ta.n_trx = tb.n_trx
group by ta.n_agr, tb.mcc
"""


print('Loading trx MCC month-by-month (cancel any hung full-period query first)...')
trx_parts = []
for m in period_months:
    m_start = m.strftime('%Y-%m-%d')
    m_end_excl = (m + pd.offsets.MonthBegin(1)).strftime('%Y-%m-%d')
    label = m.strftime('%Y-%m')
    print(f'  {label}: fetch...', flush=True)
    with imp:
        try:
            imp.execute('set MEM_LIMIT=16g')
        except Exception:
            pass
        part = imp.fetch(_sql_trx_month(m_start, m_end_excl))
    if part is None or len(part) == 0:
        print(f'  {label}: empty')
        continue
    part = part.copy()
    part['report_month'] = label
    print(f'  {label}: agr×mcc rows={len(part):,}')
    trx_parts.append(part)

if trx_parts:
    trx_mcc_month = pd.concat(trx_parts, ignore_index=True)
else:
    trx_mcc_month = pd.DataFrame(
        columns=['n_agr', 'mcc', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'report_month']
    )

trx_mcc_month['n_agr'] = trx_mcc_month['n_agr'].astype(str)
trx_mcc_month['report_month'] = trx_mcc_month['report_month'].astype(str).str[:7]
trx_mcc_month['mcc'] = trx_mcc_month['mcc'].astype(str).str.strip()
for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
    trx_mcc_month[c] = pd.to_numeric(trx_mcc_month[c], errors='coerce').fillna(0)

# primary MCC = max trx_sum per agr×month
primary_trx = (
    trx_mcc_month.sort_values(
        ['n_agr', 'report_month', 'trx_sum', 'mcc'],
        ascending=[True, True, False, True],
    )
    .groupby(['n_agr', 'report_month'], as_index=False)
    .first()
    .rename(columns={'mcc': 'mcc_trx_primary'})
    [['n_agr', 'report_month', 'mcc_trx_primary']]
)

trx_by_agr_month = (
    trx_mcc_month.groupby(['n_agr', 'report_month'], as_index=False)
    .agg(
        mcc_trx_list=('mcc', _join_unique),
        mcc_trx_cnt=('mcc', 'nunique'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
    )
)
trx_by_agr_month = trx_by_agr_month.merge(primary_trx, on=['n_agr', 'report_month'], how='left')

print('trx agr×month rows:', len(trx_by_agr_month))
print('trx agr×month with multi MCC:', int((trx_by_agr_month['mcc_trx_cnt'] > 1).sum()))
display(trx_by_agr_month.sort_values('mcc_trx_cnt', ascending=False).head(10))


## 3. Compare merchant vs trx

Join на `n_agr × report_month`, флаги multiple, overlap множеств, primary MCC.


In [ ]:
def _mcc_set(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return set()
    return {p.strip() for p in str(s).split(',') if p.strip() and p.strip() not in ('None', 'nan', 'NaN')}


cmp = merchant_month.merge(
    trx_by_agr_month,
    on=['n_agr', 'report_month'],
    how='outer',
    indicator=True,
)

cmp['mcc_merchant_cnt'] = pd.to_numeric(cmp['mcc_merchant_cnt'], errors='coerce').fillna(0).astype(int)
cmp['mcc_trx_cnt'] = pd.to_numeric(cmp['mcc_trx_cnt'], errors='coerce').fillna(0).astype(int)
cmp['trx_cnt'] = pd.to_numeric(cmp.get('trx_cnt'), errors='coerce').fillna(0)
cmp['trx_sum'] = pd.to_numeric(cmp.get('trx_sum'), errors='coerce').fillna(0)
cmp['commission_from_ops'] = pd.to_numeric(cmp.get('commission_from_ops'), errors='coerce').fillna(0)

cmp['merchant_multi'] = (cmp['mcc_merchant_cnt'] > 1).astype(int)
cmp['trx_multi'] = (cmp['mcc_trx_cnt'] > 1).astype(int)
cmp['has_merchant_mcc'] = (cmp['mcc_merchant_cnt'] > 0).astype(int)
cmp['has_trx_mcc'] = (cmp['mcc_trx_cnt'] > 0).astype(int)

sets_m = cmp['mcc_merchant_list'].map(_mcc_set)
sets_t = cmp['mcc_trx_list'].map(_mcc_set)
cmp['exact_set_match'] = [
    int(a == b and len(a) > 0) for a, b in zip(sets_m, sets_t)
]
cmp['trx_subset_of_merchant'] = [
    int(len(t) > 0 and t.issubset(m)) for m, t in zip(sets_m, sets_t)
]
cmp['merchant_subset_of_trx'] = [
    int(len(m) > 0 and m.issubset(t)) for m, t in zip(sets_m, sets_t)
]
cmp['intersection_cnt'] = [len(m & t) for m, t in zip(sets_m, sets_t)]

cmp['mcc_merchant_primary'] = cmp['mcc_merchant_list'].map(
    lambda s: str(s).split(',')[0] if pd.notna(s) and str(s).strip() else None
)
cmp['primary_match'] = (
    cmp['mcc_merchant_primary'].astype(str).str.strip()
    == cmp['mcc_trx_primary'].astype(str).str.strip()
).astype(int)
cmp.loc[cmp['mcc_trx_primary'].isna() | cmp['mcc_merchant_primary'].isna(), 'primary_match'] = 0

cmp['join_side'] = cmp['_merge'].astype(str)
cmp = cmp.drop(columns=['_merge'])

cols = [
    'report_month', 'n_agr', 'agr_id',
    'mcc_merchant_list', 'mcc_merchant_cnt', 'mcc_merchant_primary', 'merchant_multi',
    'mcc_trx_list', 'mcc_trx_cnt', 'mcc_trx_primary', 'trx_multi',
    'trx_cnt', 'trx_sum', 'commission_from_ops',
    'exact_set_match', 'trx_subset_of_merchant', 'merchant_subset_of_trx',
    'intersection_cnt', 'primary_match',
    'has_merchant_mcc', 'has_trx_mcc', 'join_side',
]
cmp = cmp[[c for c in cols if c in cmp.columns]].sort_values(['report_month', 'n_agr'])

print('compare rows (agr×month):', len(cmp))
display(cmp.head(15))
display(cmp.loc[cmp['merchant_multi'] == 1].head(10))


## 4. Summary — кто даёт меньше multiple / насколько совпадают

Рекомендация для дашборда MCC фиксируется по итогам прогона.


In [ ]:
with_both = cmp.loc[(cmp['has_merchant_mcc'] == 1) & (cmp['has_trx_mcc'] == 1)].copy()

agr_month_merchant_multi = int((cmp['merchant_multi'] == 1).sum())
agr_month_trx_multi = int((cmp['trx_multi'] == 1).sum())
delta_multi = agr_month_merchant_multi - agr_month_trx_multi

trx_universe = cmp.loc[cmp['has_trx_mcc'] == 1]
merchant_multi_among_trx = int((trx_universe['merchant_multi'] == 1).sum())
trx_multi_among_trx = int((trx_universe['trx_multi'] == 1).sum())

n_both = len(with_both)
share_exact = float(with_both['exact_set_match'].mean()) if n_both else None
share_trx_subset = float(with_both['trx_subset_of_merchant'].mean()) if n_both else None
share_primary = float(with_both['primary_match'].mean()) if n_both else None

only_merchant_multi = int(((cmp['merchant_multi'] == 1) & (cmp['trx_multi'] == 0)).sum())
only_trx_multi = int(((cmp['trx_multi'] == 1) & (cmp['merchant_multi'] == 0)).sum())
both_multi = int(((cmp['merchant_multi'] == 1) & (cmp['trx_multi'] == 1)).sum())

summary = pd.DataFrame([{
    'agr_month_rows': len(cmp),
    'agr_month_with_merchant_mcc': int(cmp['has_merchant_mcc'].sum()),
    'agr_month_with_trx_mcc': int(cmp['has_trx_mcc'].sum()),
    'agr_month_with_both': n_both,
    'agr_month_merchant_multi': agr_month_merchant_multi,
    'agr_month_trx_multi': agr_month_trx_multi,
    'delta_multi_merchant_minus_trx': delta_multi,
    'merchant_multi_among_trx_active': merchant_multi_among_trx,
    'trx_multi_among_trx_active': trx_multi_among_trx,
    'only_merchant_multi': only_merchant_multi,
    'only_trx_multi': only_trx_multi,
    'both_sides_multi': both_multi,
    'share_exact_set_match': share_exact,
    'share_trx_subset_of_merchant': share_trx_subset,
    'share_primary_mcc_match': share_primary,
}])

print('=== MCC compare summary (agr × month) ===')
display(summary.T.rename(columns={0: 'value'}))

by_month = (
    cmp.groupby('report_month', as_index=False)
    .agg(
        rows=('n_agr', 'size'),
        merchant_multi=('merchant_multi', 'sum'),
        trx_multi=('trx_multi', 'sum'),
        with_trx=('has_trx_mcc', 'sum'),
        exact_match=('exact_set_match', 'sum'),
    )
)
by_month['delta_multi'] = by_month['merchant_multi'] - by_month['trx_multi']
print('=== by month ===')
display(by_month)

if agr_month_trx_multi < agr_month_merchant_multi:
    reco = (
        'RECOMMEND: для дашборд-таблицы MCC использовать trx-слой '
        '(меньше agr×month с multiple MCC). Merchant MCC — справочно.'
    )
elif agr_month_trx_multi > agr_month_merchant_multi:
    reco = (
        'UNEXPECTED: trx multiple > merchant multiple — перепроверьте периметр trx / join на n_agr.'
    )
else:
    reco = (
        'MULTIPLE counts equal — смотрите overlap и primary_match; '
        'для объёмов всё равно предпочтителен trx.'
    )

print(reco)
print(
    f"multi merchant={agr_month_merchant_multi:,} | "
    f"multi trx={agr_month_trx_multi:,} | "
    f"delta={delta_multi:,} | "
    f"exact_match={share_exact} | "
    f"trx_subset_merchant={share_trx_subset}"
)

cmp.to_csv(out_detail_csv, index=False, encoding='utf-8-sig')
summary.to_csv(out_summary_csv, index=False, encoding='utf-8-sig')
by_month_path = output_dir / 'mcc_compare_by_month_2026_01_2026_07.csv'
by_month.to_csv(by_month_path, index=False, encoding='utf-8-sig')
print('saved:', out_detail_csv)
print('saved:', out_summary_csv)
print('saved:', by_month_path)


## 5. Как читать результат и что дальше

1. Смотрите `delta_multi_merchant_minus_trx` в summary: если **> 0**, у merchant больше multiple — как ожидалось.
2. Среди активных по trx (`with_trx`) сравните `merchant_multi_among_trx_active` vs `trx_multi_among_trx_active`.
3. Высокий `share_trx_subset_of_merchant` значит trx-коды обычно уже есть в справочнике мерчанта (справочник шире).
4. Для дашборда «Аналитика MCC» дальше используйте `sources/sql/vd_acq_mcc_month.sql` (зерно `month × mcc`, не agr).
5. `final_df` с merchant MCC **не обязательно** пересобирать.
